In [ ]:
from pyspark.sql import functions as F

In [ ]:
import os

import Upsert_Job_Offers as upsert
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# Sin cuentas de storage, catalogos, esquemas ni checkpoints hard-coded.
# En Databricks Workflows se pasan como parametros de la tarea.
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("checkpoints_root", "")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog", "job_offers")
BRONZE_SCHEMA = _cfg("bronze_schema", "bronze")
STORAGE_ACCOUNT = _cfg("storage_account")
CONTAINER = _cfg("container", "landing")

if not STORAGE_ACCOUNT:
    raise ValueError(
        "Configura el widget/variable 'storage_account' (no se versiona en el repo).")

STORAGE_BASE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CHECKPOINTS = _cfg("checkpoints_root") or f"{STORAGE_BASE}/_checkpoints"

# Streaming queries started by this notebook: the last cell waits on
# them so a failed MERGE fails the task instead of finishing green.
QUERIES = []

# The quarantine fingerprint (_row_hash) serializes timestamps with the
# session time zone: pinned to UTC so it stays stable across cluster and tests (D4).
spark.conf.set("spark.sql.session.timeZone", "UTC")


def bronze_table(name: str) -> str:
    return f"{CATALOG}.{BRONZE_SCHEMA}.{name}"


def checkpoint_path(name: str) -> str:
    return f"{CHECKPOINTS}/{name}"


def make_upsert_handler(source: str):
    """Return the foreachBatch handler of a source: quarantine + bronze upsert.

    The batch arrives with `_ingest_date` and `_source_file` added on read.
    Invalid rows go to `<catalog>.<bronze_schema>.<source>_quarantine`
    and never enter the MERGE; the rest is deduplicated by key before the
    upsert (design-upsert.md D2 and D4).
    """
    target_table = bronze_table(source)
    quarantine_table = bronze_table(f"{source}_quarantine")
    # The MERGE no longer creates the target table (the old toTable
    # did): fail when the handler is built, before the first batch,
    # with a clear message instead of failing every microbatch.
    if not spark.catalog.tableExists(target_table):
        raise ValueError(
            f"La tabla destino {target_table} no existe: creala antes de "
            "ejecutar la ingesta (el MERGE ya no la crea).")
    # The four streams run in parallel: each source uses its own temporary
    # view so a batch is never read from another source's MERGE
    # (race window between createOrReplaceTempView and spark.sql).
    batch_view = f"__batch_{source}"
    merge_sql = upsert.merge_sql(target_table, source, view_name=batch_view)

    def handler(batch_df, batch_id: int) -> None:
        if batch_df.isEmpty():
            return

        # The batch is used several times (split, quarantine and MERGE): cached
        # to avoid re-reading the landing files (D4).
        batch_df.persist()
        try:
            valid, quarantine = upsert.split_quarantine(batch_df, source)

            # Quarantine: append-only and schema-evolving; it never aborts
            # the batch nor takes part in the MERGE.
            if not quarantine.isEmpty():
                # Only the files of this batch are read from the
                # quarantine table: the identity check is bounded by the
                # batch and its cost does not grow with the table (D4).
                files = (
                    [row[0]
                     for row in batch_df.select(upsert.SOURCE_FILE_COLUMN)
                     .distinct().collect()]
                    if upsert.SOURCE_FILE_COLUMN in batch_df.columns
                    else []
                )
                if files and spark.catalog.tableExists(quarantine_table):
                    stored = (
                        spark.read.table(quarantine_table)
                        .where(F.col(upsert.SOURCE_FILE_COLUMN).isin(files))
                        .select(*upsert.QUARANTINE_IDENTITY_COLUMNS)
                    )
                else:
                    stored = quarantine.select(
                        *upsert.QUARANTINE_IDENTITY_COLUMNS).limit(0)
                new_rows = upsert.new_quarantine_rows(quarantine, stored)
                if not new_rows.isEmpty():
                    (new_rows.write
                        .format("delta")
                        .mode("append")
                        .option("mergeSchema", "true")
                        .saveAsTable(quarantine_table))

            # Only the deduplicated batch, without quarantine columns, goes to the MERGE.
            deduped = upsert.dedupe_batch(valid, source)
            if deduped.isEmpty():
                return
            deduped.createOrReplaceTempView(batch_view)
            spark.sql(merge_sql)
        finally:
            batch_df.unpersist()

    return handler


INDEED

In [ ]:
# COMMAND ----------
# 1) INDEED (Parquet)
# Source: landing/indeed/ (tu PC sube indeed_jobs_YYYYMMDD_HHMM.parquet via azcopy)
# Dest:  job_offers.bronze.indeed (+ <catalog>.<bronze_schema>.indeed_quarantine)
#
# foreachBatch + MERGE sink (spec 008): batch dedup by key and upsert.
# The current checkpoint is kept (plan A): only new files are ingested.

QUERIES.append(
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path('indeed')}/schema")
        .option("cloudFiles.useNotifications", "false")
        .load(f"{STORAGE_BASE}/indeed/")
        .withColumn("_ingest_date", F.current_date())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path('indeed')}/ckpt")
        .trigger(availableNow=True)
        .foreachBatch(make_upsert_handler("indeed"))
        .start()
)


LINKEDIN

In [ ]:
# COMMAND ----------
# 2) LINKEDIN (Parquet)
# Source: landing/linkedin/ (tu PC sube jobs_YYYYMMDD_HHMMSS.parquet via azcopy)
# Dest:  job_offers.bronze.linkedin (+ <catalog>.<bronze_schema>.linkedin_quarantine)
#
# LinkedIn es el scraper que mas tarda (hasta 10h). Su output es un unico
# jobs.parquet acumulativo que el pipeline local renombra con sufijo de
# timestamp antes de subir, para que Autoloader lo detecte como archivo nuevo.
#
# foreachBatch + MERGE sink (spec 008): batch dedup by key and upsert.
# The current checkpoint is kept (plan A): only new files are ingested.

QUERIES.append(
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path('linkedin')}/schema")
        .option("cloudFiles.useNotifications", "false")
        .load(f"{STORAGE_BASE}/linkedin/")
        .withColumn("_ingest_date", F.current_date())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path('linkedin')}/ckpt")
        .trigger(availableNow=True)
        .foreachBatch(make_upsert_handler("linkedin"))
        .start()
)


INFOJOBS

In [ ]:
# COMMAND ----------
# 3) INFOJOBS (Parquet)
# Source: landing/infojobs/ (tu PC sube offers_YYYYMMDD_HHMMSS.parquet via azcopy)
# Dest:  job_offers.bronze.infojobs (+ <catalog>.<bronze_schema>.infojobs_quarantine)
#
# InfoJobs genera muchos archivos pequenos por run (uno por ciudad/keyword).
# Autoloader los procesa todos en una sola pasada gracias al checkpoint.
#
# foreachBatch + MERGE sink (spec 008): batch dedup by key and upsert.
# The current checkpoint is kept (plan A): only new files are ingested.

QUERIES.append(
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path('infojobs')}/schema")
        .option("cloudFiles.useNotifications", "false")
        .load(f"{STORAGE_BASE}/infojobs/")
        .withColumn("_ingest_date", F.current_date())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path('infojobs')}/ckpt")
        .trigger(availableNow=True)
        .foreachBatch(make_upsert_handler("infojobs"))
        .start()
)


Multi-Site

In [ ]:
# COMMAND ----------
# 4) MULTI_SITE (Parquet)
# Source: landing/multi_site/ (tu PC sube jobs_unified_YYYYMMDD_HHMMSS.parquet)
# Dest:  job_offers.bronze.multi_site (+ <catalog>.<bronze_schema>.multi_site_quarantine)
#
# Multi-site es un merge de 4 subscrapers (irishjobs, stepstone_nl, jobs_ch,
# glassdoor). El archivo jobs_unified.parquet se renombra con timestamp antes
# de subir para que Autoloader lo detecte.
#
# foreachBatch + MERGE sink (spec 008): batch dedup by key and upsert.
# The current checkpoint is kept (plan A): only new files are ingested.

QUERIES.append(
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path('multi_site')}/schema")
        .option("cloudFiles.useNotifications", "false")
        .load(f"{STORAGE_BASE}/multi_site/")
        .withColumn("_ingest_date", F.current_date())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path('multi_site')}/ckpt")
        .trigger(availableNow=True)
        .foreachBatch(make_upsert_handler("multi_site"))
        .start()
)


In [ ]:
# Wait for the streams: awaitTermination re-raises the failure of a
# query, so a failed MERGE fails the task instead of finishing green.
# With trigger(availableNow=True) every query ends on its own.
for query in QUERIES:
    query.awaitTermination()
    print(f"Stream {query.id} finalizada: {query.status['message']}")
